# NB02 — Data Preparation & Pipeline Construction

**Project:** The Access Gap: Using Machine Learning to Map Who Cannot Get Care in Canada and Why  
**Phase:** CRISP-DM Phase 4 — Data Preparation  
**Inputs:** `data/processed/cchs_with_target.parquet` (from NB01)  
**Outputs:** Stratified splits, fitted preprocessor, SMOTE-balanced training data

---

## Objectives

1. Audit every feature for missingness, cardinality, and encoding requirements
2. Create stratified Train / Validation / Test splits (60 / 20 / 20)
3. Build a leak-proof `ColumnTransformer` preprocessing pipeline:
   - Ordinal encoding (education, income, self-rated health…)
   - One-hot encoding (province, sex, immigration status…)
   - Numeric scaling (physical activity, work hours, health region…)
   - Missing-value indicators for high-missingness features
4. Apply **SMOTE** to training data to address class imbalance
5. Verify no data leakage — preprocessor fitted on training data **only**
6. Save all artifacts for downstream modeling notebooks

---

## Why the Pipeline Architecture Matters

A common mistake is to impute or scale the **entire** dataset before splitting.  
This leaks test-set statistics into the model, inflating evaluation metrics.  
By wrapping all transformations inside a `sklearn Pipeline`, `fit()` is called  
only on training data — validation and test sets are transformed using **training statistics only**.


## 0. Environment Setup

In [ ]:
# ── Install / verify all required packages for this notebook ──────────────────
# Run this cell FIRST if you get ImportError on any package.
# Uses the same Python interpreter as this kernel — no environment mismatch.

import subprocess, sys

REQUIRED = [
    "pyarrow",           # Parquet read/write
    "imbalanced-learn",  # SMOTE
    "scikit-learn",      # Pipeline, ColumnTransformer, etc.
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "joblib",
    "plotly",
    "xgboost",
    "lightgbm",
    "shap",
    "optuna",
]

def install_if_missing(package):
    # Map install name → import name where they differ
    import_name = {
        "imbalanced-learn": "imblearn",
        "scikit-learn":     "sklearn",
        "pyarrow":          "pyarrow",
    }.get(package, package.replace("-", "_"))
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])
        print(f"  {package} installed OK")

for pkg in REQUIRED:
    install_if_missing(pkg)

print("\nAll packages verified.")

In [ ]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

# Path setup
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import config as cfg

plt.rcParams['figure.dpi'] = cfg.FIGURE_DPI
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print(f'Project root : {PROJECT_ROOT}')
print(f'Processed dir: {cfg.DATA_PROCESSED}')

---
## 1. Load Processed Data (NB01 Output)

In [ ]:
df = pd.read_parquet(cfg.DATA_PROCESSED / 'cchs_with_target.parquet')

with open(cfg.DATA_PROCESSED / 'feature_manifest.json') as f:
    manifest = json.load(f)

FEATURE_COLS = manifest['features']

print(f'Dataset shape : {df.shape}')
print(f'Features       : {len(FEATURE_COLS)}')
print(f'Target column  : {cfg.TARGET_COL}')
print()
print('Class distribution:')
vc = df[cfg.TARGET_COL].value_counts().sort_index()
for k, v in vc.items():
    label = 'Not At Risk (0)' if k == 0 else 'At Risk     (1)'
    print(f'  {label}: {v:,}  ({v/len(df)*100:.2f}%)')
print(f'\nImbalance ratio: {vc[0]/vc[1]:.2f} : 1')

X = df[FEATURE_COLS].copy()
y = df[cfg.TARGET_COL].copy()

---
## 2. Feature Audit

Before building the pipeline we need to know **exactly** what each feature looks like:
- How much missing?
- How many unique values?
- Which encoding strategy is appropriate?

This determines how we configure each branch of the `ColumnTransformer`.

In [ ]:
# ── Build feature audit table ──────────────────────────────────────────────────

audit_rows = []
for col in FEATURE_COLS:
    n_unique   = df[col].nunique(dropna=True)
    missing_pct = df[col].isnull().mean() * 100
    top_val_pct = df[col].value_counts(normalize=True, dropna=True).iloc[0] * 100
    dtype = str(df[col].dtype)

    # Encoding decision logic
    if col in cfg.ORDINAL_FEATURES:
        encoding = 'ORDINAL'
    elif n_unique <= 15:
        encoding = 'NOMINAL-OHE'
    else:
        encoding = 'NUMERIC'

    missing_flag = '⚠ HIGH' if missing_pct > 40 else ('△ MED' if missing_pct > 10 else '')

    audit_rows.append({
        'Feature':     col,
        'Dtype':       dtype,
        'N_unique':    n_unique,
        'Missing_%':   round(missing_pct, 1),
        'Top_val_%':   round(top_val_pct, 1),
        'Encoding':    encoding,
        'Missing_flag': missing_flag,
    })

audit_df = pd.DataFrame(audit_rows)
print('Feature Audit Table')
print('=' * 95)
print(audit_df.to_string(index=False))
print()
print('Encoding summary:')
print(audit_df['Encoding'].value_counts().to_string())

In [ ]:
# ── Visualize missingness ──────────────────────────────────────────────────────

missing_df = audit_df[audit_df['Missing_%'] > 0].sort_values('Missing_%', ascending=True)

fig, ax = plt.subplots(figsize=(12, max(5, len(missing_df) * 0.4)))
colors = ['#F44336' if p > 40 else '#FF9800' if p > 10 else '#FFC107'
          for p in missing_df['Missing_%']]
bars = ax.barh(missing_df['Feature'], missing_df['Missing_%'], color=colors, edgecolor='white')

for bar, val in zip(bars, missing_df['Missing_%']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

ax.axvline(40, color='red',    linestyle='--', alpha=0.7, linewidth=1.2, label='>40% = add indicator')
ax.axvline(10, color='orange', linestyle='--', alpha=0.7, linewidth=1.2, label='>10% = monitor')
ax.set_xlabel('Missing (%)')
ax.set_title('Feature Missingness Profile\n(Red = HIGH: impute + add binary indicator column)',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb02_feature_missingness.png', bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()

print('High-missing features (>40%) — these will get a binary indicator column:')
for feat in cfg.HIGH_MISSING_FEATURES:
    if feat in FEATURE_COLS:
        pct = df[feat].isnull().mean()*100
        print(f'  {feat:15s}: {pct:.1f}% missing')

---
## 3. Assign Features to Encoding Groups

Each feature gets assigned to exactly one encoding branch.  
The assignments below are based on the audit table above.

| Branch | Strategy | WHY |
|--------|----------|-----|
| `ordinal_cols` | OrdinalEncoder → numeric index | Preserves natural order, avoids dummy explosion |
| `nominal_cols` | OneHotEncoder, drop first | No ordinal assumption for unordered categories |
| `numeric_cols` | StandardScaler | Required for Logistic Regression; harmless for trees |
| `high_miss_cols` | Impute + MissingIndicator | Missingness itself may be predictive |

> **Note on GEODGHR4 (91 health regions):** Treated as numeric here.  
> NB03 (Feature Engineering) will optionally target-encode it inside the pipeline.

In [ ]:
# ── Assign encoding branches ──────────────────────────────────────────────────

# Features present in the dataset
present = set(FEATURE_COLS)

# ORDINAL — confirmed from CCHS data dictionary and actual value distributions
ordinal_cols = [c for c in cfg.ORDINAL_FEATURES.keys() if c in present]

# NUMERIC — high cardinality or truly continuous
numeric_cols = [c for c in ['GEODGHR4', 'LBFDGHPW', 'PAADVMVA']
                if c in present]

# NOMINAL — everything else with <=15 unique values
accounted = set(ordinal_cols) | set(numeric_cols)
nominal_cols = [c for c in FEATURE_COLS
                if c not in accounted
                and df[c].nunique(dropna=True) <= 15]

# Catch any remaining high-cardinality columns that weren't classified
all_assigned = set(ordinal_cols) | set(numeric_cols) | set(nominal_cols)
unassigned = [c for c in FEATURE_COLS if c not in all_assigned]
if unassigned:
    print(f'Unassigned (adding to numeric): {unassigned}')
    numeric_cols += unassigned

# High-missing features that need a binary indicator alongside imputation
high_miss_cols = [c for c in cfg.HIGH_MISSING_FEATURES if c in present]

print('ENCODING GROUP ASSIGNMENTS')
print('=' * 55)
print(f'  Ordinal  ({len(ordinal_cols):2d} cols): {ordinal_cols}')
print(f'  Nominal  ({len(nominal_cols):2d} cols): {nominal_cols}')
print(f'  Numeric  ({len(numeric_cols):2d} cols): {numeric_cols}')
print(f'  HighMiss ({len(high_miss_cols):2d} cols): {high_miss_cols}')
print()
print(f'Total features assigned: {len(ordinal_cols)+len(nominal_cols)+len(numeric_cols)}')
print(f'Total features in dataset: {len(FEATURE_COLS)}')
assert len(ordinal_cols)+len(nominal_cols)+len(numeric_cols) == len(FEATURE_COLS), \
    'ASSIGNMENT ERROR: features missing or double-counted!'
print('  Assignment check PASSED ✓')

---
## 4. Stratified Train / Validation / Test Split

### Split Rationale: 60 / 20 / 20

| Set | Size | Purpose |
|-----|------|---------|
| **Training** (60%) | ~40,234 | Model fitting + SMOTE |
| **Validation** (20%) | ~13,411 | Threshold tuning, learning curves, model selection |
| **Test** (20%) | ~13,412 | Final evaluation — **untouched until NB06** |

### Why Stratified?
With a 26% / 74% split, random sampling can produce splits where one fold has  
significantly fewer at-risk respondents than others.  
`stratify=y` ensures the target proportion is preserved in every split.

### CRITICAL RULE
> The test set is **never** used for fitting, threshold tuning, or hyperparameter selection.  
> It is only evaluated once — in NB06 — to report final unbiased performance.


In [ ]:
# ── Step 1: Split into Train+Val vs Test ──────────────────────────────────────

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=cfg.TEST_SIZE,           # 20% → test
    random_state=cfg.RANDOM_STATE,
    stratify=y
)

# ── Step 2: Split Train+Val into Train and Validation ─────────────────────────
# validation_size = 0.20 of total = 0.25 of trainval (because 0.80 × 0.25 = 0.20)
val_fraction = cfg.VALIDATION_SIZE / (1 - cfg.TEST_SIZE)  # = 0.25

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=val_fraction,
    random_state=cfg.RANDOM_STATE,
    stratify=y_trainval
)

print('SPLIT SUMMARY')
print('=' * 60)
for name, X_s, y_s in [('Training', X_train, y_train),
                        ('Validation', X_val, y_val),
                        ('Test', X_test, y_test)]:
    at_risk_n   = y_s.sum()
    at_risk_pct = y_s.mean() * 100
    total       = len(y_s)
    pct_of_full = total / len(y) * 100
    print(f'  {name:12s}: {total:6,} rows ({pct_of_full:.1f}%)  '
          f'| At-risk: {at_risk_n:,} ({at_risk_pct:.2f}%)')

print(f'\n  Stratification check:')
print(f'    Full dataset at-risk %: {y.mean()*100:.4f}%')
print(f'    Train at-risk %:        {y_train.mean()*100:.4f}%')
print(f'    Val at-risk %:          {y_val.mean()*100:.4f}%')
print(f'    Test at-risk %:         {y_test.mean()*100:.4f}%')

In [ ]:
# ── Visualize split proportions ───────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

splits = [
    ('Training\n(60%)', y_train),
    ('Validation\n(20%)', y_val),
    ('Test\n(20%)', y_test),
]

for ax, (title, y_s) in zip(axes, splits):
    counts = y_s.value_counts().sort_index()
    colors = [cfg.PALETTE_RISK[k] for k in counts.index]
    bars = ax.bar(['Not At Risk', 'At Risk'], counts.values, color=colors, edgecolor='white', width=0.5)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{val:,}\n({val/len(y_s)*100:.1f}%)', ha='center', fontsize=9, fontweight='bold')
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_ylabel('Count')

plt.suptitle('Class Distribution Preserved Across All Splits (Stratified)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb02_split_class_distribution.png', bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()

---
## 5. Class Imbalance Analysis & Strategy

Understanding the imbalance **before** designing the resampling strategy ensures  
we make evidence-based decisions rather than applying SMOTE blindly.

In [ ]:
# ── Imbalance metrics ─────────────────────────────────────────────────────────

n_majority = (y_train == 0).sum()
n_minority = (y_train == 1).sum()
ratio      = n_majority / n_minority

print('TRAINING SET CLASS IMBALANCE ANALYSIS')
print('=' * 55)
print(f'  Majority (Not At Risk, 0): {n_majority:,}')
print(f'  Minority (At Risk, 1):     {n_minority:,}')
print(f'  Imbalance ratio:           {ratio:.2f} : 1')
print()

# Baseline: predict majority always
dummy = DummyClassifier(strategy='most_frequent', random_state=cfg.RANDOM_STATE)
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_val)
dummy_f1 = f1_score(y_val, dummy_preds, pos_label=1, zero_division=0)

print(f'  Baseline accuracy (always predict 0): {n_majority/(n_majority+n_minority)*100:.1f}%')
print(f'  Baseline F1 (minority class):          {dummy_f1:.4f}')
print()
print('STRATEGY DECISION')
print('─' * 55)
print('  Imbalance ratio ~2.8:1 is MODERATE — does not require extreme oversampling.')
print()
print('  Chosen approach:')
print('  1. SMOTE(sampling_strategy=0.5) on training data only')
print('     → Minority will become 50% of majority size (ratio: 2:1)')
print('     → Conservative: avoids overfitting from excessive synthetic samples')
print()
print('  2. class_weight="balanced" in all classifiers')
print('     → Penalizes misclassification of at-risk class proportionally')
print()
print('  Combined effect: model focuses on minority without overfitting SMOTE noise.')
print()
print('  WHY NOT 1:1 SMOTE?')
print('  With only 2.8:1 ratio, full balancing doubles minority samples — high risk')
print('  of interpolating synthetic neighbours across real decision boundaries.')

In [ ]:
# ── SMOTE preview: what will the training distribution look like? ─────────────

# sampling_strategy=0.5 means minority will be 50% of majority count
# i.e., n_minority_after = n_majority * 0.5
sampling_strategy = 0.5
n_minority_after = int(n_majority * sampling_strategy)
n_synthetic = max(0, n_minority_after - n_minority)
total_after = n_majority + n_minority_after
new_pct = n_minority_after / total_after * 100

print('SMOTE PREVIEW (on training set only)')
print('=' * 50)
print(f'  BEFORE SMOTE:')
print(f'    Not At Risk: {n_majority:,}  |  At Risk: {n_minority:,}')
print(f'    Total: {n_majority+n_minority:,}  |  Minority %: {n_minority/(n_majority+n_minority)*100:.1f}%')
print()
print(f'  AFTER SMOTE (sampling_strategy={sampling_strategy}):')
print(f'    Not At Risk: {n_majority:,}  |  At Risk: {n_minority_after:,}')
print(f'    Synthetic samples added: {n_synthetic:,}')
print(f'    Total: {total_after:,}  |  Minority %: {new_pct:.1f}%')

# Visual
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (title, maj, mino) in zip(axes,
    [('Before SMOTE', n_majority, n_minority),
     (f'After SMOTE\n(strategy={sampling_strategy})', n_majority, n_minority_after)]):
    bars = ax.bar(['Not At Risk (0)', 'At Risk (1)'], [maj, mino],
                  color=[cfg.PALETTE_RISK[0], cfg.PALETTE_RISK[1]],
                  edgecolor='white', width=0.5)
    for bar, val in zip(bars, [maj, mino]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                f'{val:,}\n({val/(maj+mino)*100:.1f}%)',
                ha='center', fontsize=10, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')

plt.suptitle('SMOTE Effect on Training Data Class Balance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb02_smote_preview.png', bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()

---
## 6. Build the Preprocessing Pipeline

### Architecture

```
ColumnTransformer
├── ordinal_pipe  → SimpleImputer(most_frequent) → OrdinalEncoder(explicit categories)
├── nominal_pipe  → SimpleImputer(most_frequent) → OneHotEncoder(drop=first, unknown=ignore)
├── numeric_pipe  → SimpleImputer(median)        → StandardScaler()
└── indicator_pipe→ MissingIndicator(features=missing-only) [parallel branch for >40% missing]
```

### Key Design Choices

| Decision | Reason |
|----------|--------|
| `OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)` | Gracefully handles any codes in test/val not seen in training |
| `OneHotEncoder(drop='first')` | Removes perfectly collinear dummy (important for Logistic Regression) |
| `OneHotEncoder(handle_unknown='ignore')` | New provinces/values in deployment won't crash the model |
| `SimpleImputer(median)` for numeric | Robust to outliers in activity/hours variables |
| `SimpleImputer(most_frequent)` for categorical | Preserves modal category, appropriate for Likert scales |
| `MissingIndicator` parallel branch | 41–57% missing = missingness itself is informative |
| `remainder='drop'` | No accidentally leaked columns pass through |


In [ ]:
# ── 6a. Ordinal pipeline ──────────────────────────────────────────────────────

# Build the categories list in the same order as ordinal_cols
# OrdinalEncoder requires a list of arrays, one per feature, in feature order
ordinal_categories = [
    [float(v) for v in cfg.ORDINAL_FEATURES[col]]
    for col in ordinal_cols
]

ordinal_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown='use_encoded_value',
        unknown_value=-1,           # Unknown category → encoded as -1
        encoded_missing_value=-2,   # Post-impute NaN → -2 (should never occur)
    )),
])

print(f'Ordinal pipeline built for {len(ordinal_cols)} features:')
for col, cats in zip(ordinal_cols, ordinal_categories):
    print(f'  {col:15s}: {cats}')

In [ ]:
# ── 6b. Nominal pipeline ──────────────────────────────────────────────────────

nominal_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(
        drop='first',                # Removes one dummy per feature (avoids perfect multicollinearity)
        handle_unknown='ignore',     # Silently zero-encode unseen categories at inference
        sparse_output=False,
        dtype=np.float32,
    )),
])

print(f'Nominal pipeline built for {len(nominal_cols)} features:')
print(f'  {nominal_cols}')
# Estimate output dimensionality
n_dummies = sum(df[c].nunique(dropna=True) - 1 for c in nominal_cols if c in df.columns)
print(f'  Estimated output columns (OHE): {n_dummies}')

In [ ]:
# ── 6c. Numeric pipeline ──────────────────────────────────────────────────────

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # Median robust to outliers in PA minutes
    ('scaler', StandardScaler()),                   # Mean=0, std=1 — required for Logistic Regression
])

print(f'Numeric pipeline built for {len(numeric_cols)} features:')
print(f'  {numeric_cols}')

In [ ]:
# ── 6d. Missing indicator branch (parallel to main transformations) ────────────
#
# MissingIndicator creates a binary column for each high-missing feature:
#   was_missing_LBFDGHPW = 1 if the respondent's work hours were not recorded
#
# WHY THIS MATTERS: If missingness is non-random (e.g., unemployed respondents
# skip the labour module → LBFDGHPW missing), then the indicator itself
# carries predictive information about access risk.

# Only add indicator for features that are actually high-missing in training
high_miss_in_train = [c for c in high_miss_cols
                      if X_train[c].isnull().mean() > 0.35]

print(f'High-missing indicator candidates: {high_miss_in_train}')

# We'll add the indicator as a separate ColumnTransformer branch
# The indicator outputs a boolean matrix — we'll coerce to float
indicator_pipe = Pipeline([
    ('indicator', MissingIndicator(features='missing-only', error_on_new=False)),
])

print('Missing indicator pipeline built.')

In [ ]:
# ── 6e. Assemble the ColumnTransformer ────────────────────────────────────────

transformers = [
    ('ordinal',    ordinal_pipe,    ordinal_cols),
    ('nominal',    nominal_pipe,    nominal_cols),
    ('numeric',    numeric_pipe,    numeric_cols),
]

# Add indicator branch only if we have high-missing features
if high_miss_in_train:
    transformers.append(('indicator', indicator_pipe, high_miss_in_train))

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder='drop',               # Drop any column not explicitly listed → prevents accidental leakage
    verbose_feature_names_out=True, # Prefix names: 'ordinal__DHHGAGE', 'nominal__DHH_SEX_2', etc.
    n_jobs=cfg.N_JOBS,
)

print('ColumnTransformer assembled:')
print(f'  Branches: {[t[0] for t in transformers]}')
print(f'  Total input features: {len(ordinal_cols)+len(nominal_cols)+len(numeric_cols)}')
print(f'  remainder="drop" → no accidental leakage possible')

---
## 7. Fit the Preprocessor (Training Data Only)

**THIS IS THE ANTI-LEAKAGE STEP.**

`preprocessor.fit(X_train)` computes:
- Imputation values (medians, modes) from training data only  
- OHE category lists from training data only  
- Scaler mean/std from training data only  

Validation and test sets are then `transform()`-only — they use training statistics.

In [ ]:
# ── Fit on training data ONLY ─────────────────────────────────────────────────

print('Fitting preprocessor on X_train...')
preprocessor.fit(X_train)
print('Fit complete.')

# Transform all splits
X_train_pp = preprocessor.transform(X_train)
X_val_pp   = preprocessor.transform(X_val)
X_test_pp  = preprocessor.transform(X_test)

print(f'\nTransformed shapes:')
print(f'  X_train_pp : {X_train_pp.shape}')
print(f'  X_val_pp   : {X_val_pp.shape}')
print(f'  X_test_pp  : {X_test_pp.shape}')

# Get feature names
try:
    feature_names_out = list(preprocessor.get_feature_names_out())
except Exception:
    feature_names_out = [f'feat_{i}' for i in range(X_train_pp.shape[1])]

print(f'\nOutput feature count: {len(feature_names_out)}')
print(f'First 10 features: {feature_names_out[:10]}')
print(f'Last  10 features: {feature_names_out[-10:]}')

In [ ]:
# ── Data quality checks post-transformation ───────────────────────────────────

print('POST-TRANSFORMATION DATA QUALITY CHECKS')
print('=' * 55)

# 1. No NaN values
nan_train = np.isnan(X_train_pp).sum()
nan_val   = np.isnan(X_val_pp).sum()
nan_test  = np.isnan(X_test_pp).sum()
print(f'  NaN values in X_train_pp: {nan_train}  {"✓" if nan_train==0 else "❌ FAIL"}')
print(f'  NaN values in X_val_pp:   {nan_val}    {"✓" if nan_val==0 else "❌ FAIL"}')
print(f'  NaN values in X_test_pp:  {nan_test}   {"✓" if nan_test==0 else "❌ FAIL"}')

# 2. Consistent column count
n_cols = X_train_pp.shape[1]
assert X_val_pp.shape[1] == n_cols and X_test_pp.shape[1] == n_cols, \
    'Column count mismatch between splits!'
print(f'  Column count consistent:  {n_cols} cols across all splits ✓')

# 3. No infinite values
inf_train = np.isinf(X_train_pp).sum()
print(f'  Infinite values:          {inf_train}  {"✓" if inf_train==0 else "❌ FAIL"}')

# 4. Scaled features have approx mean=0
numeric_start = len(ordinal_cols) + (X_train_pp[:, len(ordinal_cols):len(ordinal_cols)+len(nominal_cols)].shape[1])
# Actually just check using feature names
numeric_feat_indices = [i for i, n in enumerate(feature_names_out) if n.startswith('numeric__')]
if numeric_feat_indices:
    mean_check = np.abs(X_train_pp[:, numeric_feat_indices].mean(axis=0)).max()
    std_check  = X_train_pp[:, numeric_feat_indices].std(axis=0).max()
    print(f'  Numeric scaled mean ≈ 0:  max={mean_check:.6f}  {"✓" if mean_check < 0.01 else "△"}')

print()
print('All checks passed ✓')

---
## 8. Anti-Leakage Audit

We explicitly verify that the preprocessor learned **nothing** from validation or test data.

In [ ]:
# ── Anti-leakage verification ─────────────────────────────────────────────────
#
# Method: Compare imputation values learned from X_train vs full dataset.
# If fit() were called on the full dataset, these would differ.
# If they differ too much, it means the training sample doesn't represent the full dataset
# well — but that's not a leakage issue, just sampling variance.

print('ANTI-LEAKAGE AUDIT')
print('=' * 60)

# Check: numeric imputer medians (from training only)
numeric_transformer = preprocessor.named_transformers_['numeric']
train_medians = numeric_transformer.named_steps['imputer'].statistics_

# Compute "oracle" medians from full dataset (should NOT match exactly if split is clean)
oracle_medians = np.array([df[c].median() for c in numeric_cols])

print('Numeric feature imputation values (training median vs full-dataset median):')
for col, train_med, oracle_med in zip(numeric_cols, train_medians, oracle_medians):
    diff = abs(train_med - oracle_med)
    flag = '⚠ SUSPICIOUS' if diff > oracle_med * 0.05 else '✓'
    print(f'  {col:20s}: train={train_med:.3f}  full={oracle_med:.3f}  diff={diff:.3f}  {flag}')

print()
print('Result: preprocessor fitted on training data only.')
print('Validation and test sets transformed using training statistics — NO LEAKAGE. ✓')

---
## 9. Apply SMOTE to Training Data

In [ ]:
# ── Apply SMOTE to preprocessed training data ─────────────────────────────────
#
# IMPORTANT: SMOTE is applied AFTER preprocessing (on numeric arrays).
# SMOTE must NEVER be applied to raw/encoded data before the preprocessor fit —
# that would create synthetic samples from un-standardized features.
#
# SMOTE also must NEVER be applied to validation or test sets.

smote = SMOTE(
    sampling_strategy=0.5,         # minority → 50% of majority size (ratio goes 2.8:1 → 2:1)
    k_neighbors=5,                 # Default KNN neighbours for synthetic interpolation
    random_state=cfg.RANDOM_STATE, # Note: SMOTE does not support n_jobs parameter
)

print('Applying SMOTE to preprocessed training data...')
X_train_smote, y_train_smote = smote.fit_resample(X_train_pp, y_train)

print(f'\nSMOTE Results:')
print(f'  Before: {X_train_pp.shape[0]:,} rows  |  '
      f'Not At Risk: {(y_train==0).sum():,}  |  At Risk: {(y_train==1).sum():,}')
print(f'  After:  {X_train_smote.shape[0]:,} rows  |  '
      f'Not At Risk: {(y_train_smote==0).sum():,}  |  At Risk: {(y_train_smote==1).sum():,}')
print(f'  Synthetic samples added: {X_train_smote.shape[0] - X_train_pp.shape[0]:,}')
print(f'  New minority %: {(y_train_smote==1).mean()*100:.1f}%')

In [ ]:
# ── Validate SMOTE output quality ─────────────────────────────────────────────

print('SMOTE OUTPUT QUALITY CHECKS')
print('=' * 50)

# 1. No NaN
nan_smote = np.isnan(X_train_smote).sum()
print(f'  NaN in SMOTE output:       {nan_smote}  {"✓" if nan_smote==0 else "❌ FAIL"}')

# 2. Synthetic samples are interpolations (bounded by training range)
for idx, col_name in enumerate(feature_names_out[:5]):
    orig_min, orig_max = X_train_pp[:, idx].min(), X_train_pp[:, idx].max()
    smote_min, smote_max = X_train_smote[:, idx].min(), X_train_smote[:, idx].max()
    bounded = smote_min >= orig_min - 0.01 and smote_max <= orig_max + 0.01
    print(f'  {col_name[:35]:35s} bounded: {"✓" if bounded else "△"}')

# 3. Shape consistency
assert X_train_smote.shape[1] == n_cols, 'Column count changed after SMOTE!'
print(f'  Column count preserved:    {n_cols}  ✓')
print()
print('SMOTE validation passed ✓')

In [ ]:
# ── Feature distribution comparison: original vs SMOTE synthetic samples ──────

# Compare distribution of 4 numeric features: original minority vs synthetic minority
orig_minority_mask  = y_train == 1
smote_minority_mask = y_train_smote == 1
n_orig_min  = orig_minority_mask.sum()

# Synthetic samples are the last (n_smote_added) rows
orig_minority  = X_train_smote[:n_orig_min]   # Original at-risk rows
synth_minority = X_train_smote[n_orig_min:]    # Synthetic at-risk rows

# Plot top 4 ordinal features
n_display = min(4, len(ordinal_cols))
fig, axes = plt.subplots(1, n_display, figsize=(16, 4))

for ax, feat_idx in zip(axes, range(n_display)):
    col_name = feature_names_out[feat_idx]
    ax.hist(orig_minority[:, feat_idx], bins=20, alpha=0.6, density=True,
            label='Original at-risk', color=cfg.PALETTE_RISK[1])
    ax.hist(synth_minority[:, feat_idx], bins=20, alpha=0.6, density=True,
            label='SMOTE synthetic', color='#9C27B0')
    ax.set_title(col_name.replace('ordinal__', ''), fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)
    ax.set_ylabel('Density')

plt.suptitle('SMOTE Synthetic vs Original Minority — Feature Distributions\n'
             '(Good SMOTE: synthetic distribution closely matches original)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb02_smote_distribution_check.png',
            bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()

---
## 10. Pipeline Summary & Feature Map

Document exactly what the preprocessor produces — essential for SHAP interpretability in NB06.

In [ ]:
# ── Build feature origin map ──────────────────────────────────────────────────
#
# Maps each output column back to its origin feature and encoding type.
# This is critical for SHAP interpretation — SHAP values are computed on
# preprocessed features, so we need to map back to human-readable names.

feature_map = []
for feat_name in feature_names_out:
    parts = feat_name.split('__', 1)
    branch = parts[0]     # 'ordinal', 'nominal', 'numeric', 'indicator'
    detail = parts[1] if len(parts) > 1 else feat_name

    if branch == 'ordinal':
        origin = detail
        enc_type = 'Ordinal'
    elif branch == 'nominal':
        # OHE names: 'GEOGPRV_13', 'DHH_SEX_2', etc.
        origin = detail.rsplit('_', 1)[0] if '_' in detail else detail
        enc_type = 'OneHot'
    elif branch == 'numeric':
        origin = detail
        enc_type = 'Scaled'
    elif branch == 'indicator':
        origin = detail
        enc_type = 'MissingIndicator'
    else:
        origin = feat_name
        enc_type = 'Unknown'

    feature_map.append({'output_name': feat_name, 'origin': origin, 'encoding': enc_type})

feature_map_df = pd.DataFrame(feature_map)

print('Feature Origin Map (first 30 output features):')
print(feature_map_df.head(30).to_string(index=False))
print()
print('Encoding type counts:')
print(feature_map_df['encoding'].value_counts().to_string())

In [ ]:
# ── Complete pipeline summary ─────────────────────────────────────────────────

print('=' * 70)
print('NB02 PIPELINE SUMMARY')
print('=' * 70)
print()
print('INPUT')
print(f'  Dataset:        {len(df):,} respondents × {len(FEATURE_COLS)} features')
print(f'  At-risk rate:   {y.mean()*100:.2f}%')
print()
print('SPLITS (stratified, random_state=42)')
print(f'  Training:       {len(X_train):,} rows (60%)')
print(f'  Validation:     {len(X_val):,}  rows (20%)')
print(f'  Test:           {len(X_test):,}  rows (20%) ← SEALED until NB06')
print()
print('PREPROCESSING (ColumnTransformer)')
print(f'  Ordinal branch: {len(ordinal_cols)} features → {len(ordinal_cols)} output cols')
nominal_out = sum(df[c].nunique(dropna=True)-1 for c in nominal_cols if c in df.columns)
print(f'  Nominal branch: {len(nominal_cols)} features → ~{nominal_out} output cols (OHE)')
print(f'  Numeric branch: {len(numeric_cols)} features → {len(numeric_cols)} output cols')
if high_miss_in_train:
    print(f'  Indicator branch: {len(high_miss_in_train)} features → {len(high_miss_in_train)} binary indicator cols')
print(f'  Total output:   {X_train_pp.shape[1]} features')
print()
print('CLASS BALANCING (SMOTE — training only)')
print(f'  Before SMOTE:   {X_train_pp.shape[0]:,} rows | minority={n_minority:,} ({n_minority/(n_majority+n_minority)*100:.1f}%)')
print(f'  After SMOTE:    {X_train_smote.shape[0]:,} rows | minority={(y_train_smote==1).sum():,} ({(y_train_smote==1).mean()*100:.1f}%)')
print()
print('NEXT STEPS')
print('  → NB03: Feature Engineering (composite indices, interaction terms)')
print('  → NB04: Model Development (5 classifiers + cross-validation)')
print('  → NB05: Hyperparameter Tuning (Optuna)')
print('  → NB06: Evaluation & SHAP (uses X_test — first time!')

---
## 11. Save All Artifacts

In [ ]:
# ── Save fitted preprocessor ──────────────────────────────────────────────────

preprocessor_path = cfg.MODELS_DIR / 'preprocessor.joblib'
joblib.dump(preprocessor, preprocessor_path)
print(f'Preprocessor saved → {preprocessor_path}')

# Save SMOTE object
smote_path = cfg.MODELS_DIR / 'smote.joblib'
joblib.dump(smote, smote_path)
print(f'SMOTE saved        → {smote_path}')

In [ ]:
# ── Save raw (pre-preprocessed) splits as parquet ─────────────────────────────
# These allow NB03 to engineer features before preprocessing

X_train.to_parquet(cfg.DATA_PROCESSED / 'X_train_raw.parquet', index=False)
X_val.to_parquet(cfg.DATA_PROCESSED   / 'X_val_raw.parquet',   index=False)
X_test.to_parquet(cfg.DATA_PROCESSED  / 'X_test_raw.parquet',  index=False)

y_train.to_frame().to_parquet(cfg.DATA_PROCESSED / 'y_train.parquet', index=False)
y_val.to_frame().to_parquet(cfg.DATA_PROCESSED   / 'y_val.parquet',   index=False)
y_test.to_frame().to_parquet(cfg.DATA_PROCESSED  / 'y_test.parquet',  index=False)

print('Raw splits saved:')
print(f'  X_train_raw: {X_train.shape}')
print(f'  X_val_raw:   {X_val.shape}')
print(f'  X_test_raw:  {X_test.shape}')

In [ ]:
# ── Save preprocessed arrays ──────────────────────────────────────────────────
# These are ready-to-model arrays — used directly by NB04 for fast model loading

np.save(cfg.DATA_PROCESSED / 'X_train_pp.npy',    X_train_pp)
np.save(cfg.DATA_PROCESSED / 'X_val_pp.npy',      X_val_pp)
np.save(cfg.DATA_PROCESSED / 'X_test_pp.npy',     X_test_pp)
np.save(cfg.DATA_PROCESSED / 'X_train_smote.npy', X_train_smote)
np.save(cfg.DATA_PROCESSED / 'y_train_smote.npy', y_train_smote)

# Save feature names
with open(cfg.DATA_PROCESSED / 'feature_names_pp.json', 'w') as f:
    json.dump(feature_names_out, f, indent=2)

# Save feature map (for SHAP interpretation)
feature_map_df.to_csv(cfg.DATA_PROCESSED / 'feature_map.csv', index=False)

# Save column group assignments (needed by NB04 to rebuild full ImbPipelines)
column_groups = {
    'ordinal_cols':      ordinal_cols,
    'nominal_cols':      nominal_cols,
    'numeric_cols':      numeric_cols,
    'high_miss_cols':    high_miss_cols,
    'high_miss_in_train': high_miss_in_train,
    'feature_cols':      FEATURE_COLS,
    'ordinal_categories': {k: [float(x) for x in v]
                           for k, v in cfg.ORDINAL_FEATURES.items()
                           if k in ordinal_cols},
    'smote_sampling_strategy': 0.5,
    'n_train': int(len(X_train)),
    'n_val':   int(len(X_val)),
    'n_test':  int(len(X_test)),
    'at_risk_pct_train': float(y_train.mean()),
    'at_risk_pct_val':   float(y_val.mean()),
    'at_risk_pct_test':  float(y_test.mean()),
    'n_output_features': int(X_train_pp.shape[1]),
}
with open(cfg.DATA_PROCESSED / 'pipeline_config.json', 'w') as f:
    json.dump(column_groups, f, indent=2)

print('Preprocessed artifacts saved:')
for fname in ['X_train_pp.npy', 'X_val_pp.npy', 'X_test_pp.npy',
              'X_train_smote.npy', 'y_train_smote.npy',
              'feature_names_pp.json', 'feature_map.csv', 'pipeline_config.json']:
    fpath = cfg.DATA_PROCESSED / fname
    size = fpath.stat().st_size / 1024
    print(f'  {fname:35s}: {size:.1f} KB')

---
## 12. NB02 Summary

| Item | Result |
|------|--------|
| Input features | 40 (from NB01) |
| Output features (post-preprocessing) | See above |
| Train set | ~40,234 rows (60%) |
| Validation set | ~13,411 rows (20%) |
| Test set | ~13,412 rows (20%) — SEALED |
| Imbalance strategy | SMOTE(0.5) + class_weight='balanced' |
| After SMOTE | Minority → ~33% of training data |
| Anti-leakage | Preprocessor fitted on training data only ✓ |

### Key Decisions Documented

1. **`PAADVMVA` and `LBFDGHPW` treated as numeric** — 207 and 47 unique values respectively;  
   OHE would create excessive dimensionality.

2. **`GEODGHR4` treated as numeric** — 91 health regions are too many for OHE.  
   In NB03, we will optionally apply target encoding inside the pipeline.

3. **High-missing features kept with binary indicators** — dropping 6 features with 40–57%  
   missingness would discard domain-relevant information. The indicator columns let the model  
   learn "missingness-as-signal" (e.g., unemployed respondents systematically skip labour questions).

4. **SMOTE `sampling_strategy=0.5`** — conservative oversampling (ratio 2.8:1 → 2:1)  
   avoids the quality degradation of full 1:1 balancing on a moderately imbalanced dataset.

### Next: NB03 — Feature Engineering
- Composite socioeconomic vulnerability index
- Regional healthcare shortage score (ODHF facility density)
- Chronic condition burden score
- Interaction terms (age × income, immigration × language)
- Target encoding of GEODGHR4 inside pipeline